# Plant Disease Detection using Convolutional Neural Networks

A comprehensive deep learning project for automated plant disease classification using the PlantVillage dataset.

**Dataset**: PlantVillage (38 classes across 14 crop species)
**Framework**: TensorFlow/Keras
**Architecture**: Custom CNN with BatchNorm and Dropout

## 1. Setup and Imports

In [ ]:
# Install dependencies
!pip install tensorflow keras numpy pandas matplotlib seaborn scikit-learn pillow opencv-python kaggle -q

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# Setup style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# Configuration
DATASET_PATH = os.path.expanduser('~/plant_disease_data')
MODEL_SAVE_PATH = os.path.expanduser('~/models')
LOGS_PATH = os.path.expanduser('~/logs')

IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32
EPOCHS = 100
INITIAL_LEARNING_RATE = 0.001

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(LOGS_PATH, exist_ok=True)

print('Directories created successfully!')

## 2. Download and Load Dataset

In [ ]:
# For Google Colab - Setup Kaggle API
# Uncomment below if using Colab
"""
from google.colab import files
files.upload()  # Upload kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
"""

In [ ]:
# Download dataset from Kaggle
import subprocess

if not os.path.exists(os.path.join(DATASET_PATH, 'PlantVillage')):
    print('Downloading PlantVillage dataset from Kaggle...')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'emmarex/plantdisease', 
                    '-p', DATASET_PATH, '--unzip'], check=True)
    print('Dataset downloaded and extracted!')
else:
    print('Dataset already exists!')

# Find dataset directory
dataset_dir = DATASET_PATH
if os.path.exists(os.path.join(DATASET_PATH, 'PlantVillage')):
    dataset_dir = os.path.join(DATASET_PATH, 'PlantVillage')

print(f'Using dataset from: {dataset_dir}')

## 3. Exploratory Data Analysis

In [ ]:
# Analyze dataset structure
class_folders = sorted([d for d in os.listdir(dataset_dir) 
                        if os.path.isdir(os.path.join(dataset_dir, d))])

print(f'Total Classes: {len(class_folders)}')
print(f'Sample Classes: {class_folders[:5]}')

# Count images per class
class_counts = {}
for class_name in class_folders:
    class_path = os.path.join(dataset_dir, class_name)
    num_images = len([f for f in os.listdir(class_path) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    class_counts[class_name] = num_images

df_counts = pd.DataFrame(list(class_counts.items()), columns=['Class', 'Count'])
print(f'\nTotal Images: {df_counts["Count"].sum()}')
print(f'Average Images per Class: {df_counts["Count"].mean():.2f}')
print(f'Min: {df_counts["Count"].min()}, Max: {df_counts["Count"].max()}')

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# All classes
axes[0].barh(df_counts['Class'], df_counts['Count'], color='skyblue', edgecolor='navy')
axes[0].set_xlabel('Number of Images')
axes[0].set_title('Dataset Class Distribution')
axes[0].grid(axis='x', alpha=0.3)

# Statistics
axes[1].text(0.1, 0.8, f'Total Classes: {len(class_counts)}', fontsize=12, transform=axes[1].transAxes)
axes[1].text(0.1, 0.6, f'Total Images: {df_counts["Count"].sum()}', fontsize=12, transform=axes[1].transAxes)
axes[1].text(0.1, 0.4, f'Average per Class: {df_counts["Count"].mean():.2f}', fontsize=12, transform=axes[1].transAxes)
axes[1].text(0.1, 0.2, f'Range: {df_counts["Count"].min()} - {df_counts["Count"].max()}', fontsize=12, transform=axes[1].transAxes)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(4, 4, figsize=(14, 12))
axes = axes.ravel()

for idx, class_name in enumerate(class_folders[:16]):
    class_path = os.path.join(dataset_dir, class_name)
    sample_image = None
    
    for file in os.listdir(class_path):
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            sample_image = os.path.join(class_path, file)
            break
    
    if sample_image:
        img = load_img(sample_image)
        axes[idx].imshow(img)
    
    axes[idx].set_title(class_name.split('___')[-1], fontsize=10)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 4. Data Loading and Preprocessing

In [ ]:
# Load images and labels
print('Loading images...')
images = []
labels = []

for class_idx, class_name in enumerate(class_folders):
    class_path = os.path.join(dataset_dir, class_name)
    image_files = [f for f in os.listdir(class_path) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if (class_idx + 1) % 5 == 0:
        print(f'  Processing class {class_idx + 1}/{len(class_folders)}: {class_name}')
    
    for image_file in image_files:
        try:
            image_path = os.path.join(class_path, image_file)
            image = load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
            image_array = img_to_array(image)
            images.append(image_array)
            labels.append(class_idx)
        except Exception as e:
            pass

images = np.array(images)
labels = np.array(labels)

print(f'\nLoaded {len(images)} images')
print(f'Shape: {images.shape}')
print(f'Labels shape: {labels.shape}')

In [ ]:
# Normalize images
print('Normalizing images...')
images = images / 255.0
print(f'Image range: {images.min():.2f} - {images.max():.2f}')

# Convert labels to one-hot encoding
labels_onehot = tf.keras.utils.to_categorical(labels, len(class_folders))
print(f'One-hot labels shape: {labels_onehot.shape}')

In [ ]:
# Split data into train, validation, and test sets
print('Splitting data...')

# First split: train+val vs test (80% vs 20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    images, labels_onehot, test_size=0.2, random_state=42, stratify=labels
)

# Second split: train vs val (80% vs 20% of remaining)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=np.argmax(y_temp, axis=1)
)

print(f'Training set: {X_train.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Test set: {X_test.shape}')

In [ ]:
# Visualize data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.ravel()

sample_image = X_train[0].reshape((1,) + X_train[0].shape)
i = 0

for batch in datagen.flow(sample_image, batch_size=1):
    axes[i].imshow(batch[0])
    axes[i].set_title(f'Augmentation {i+1}')
    axes[i].axis('off')
    i += 1
    if i >= 9:
        break

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Build Model Architecture

In [ ]:
# Build Custom CNN Model
print('Building CNN model...')

model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', 
                  kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.5),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.5),
    
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.5),
    
    # Global Average Pooling
    layers.GlobalAveragePooling2D(),
    
    # Dense layers
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    # Output layer
    layers.Dense(len(class_folders), activation='softmax')
], name='PlantDiseaseDetectionCNN')

print(f'Model built successfully with {model.count_params():,} parameters')
model.summary()

## 6. Compile and Train Model

In [ ]:
# Compile model
optimizer = tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE)
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')]
)

print('Model compiled!')

In [ ]:
# Define callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        os.path.join(MODEL_SAVE_PATH, 'best_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    TensorBoard(
        log_dir=LOGS_PATH,
        histogram_freq=1,
        write_graph=True
    )
]

print('Callbacks configured!')

In [ ]:
# Train model
print('Starting training...\n')

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print('\nTraining completed!')

## 7. Evaluate Model

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Training Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Validation Loss', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final Training Accuracy: {history.history["accuracy"][-1]:.4f}')
print(f'Final Validation Accuracy: {history.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Evaluate on test set
print('Evaluating on test set...')
test_loss, test_acc, test_top5 = model.evaluate(X_test, y_test, verbose=0)

print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Top-5 Accuracy: {test_top5:.4f}')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
# Generate predictions
print('Generating predictions...')
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print('Predictions generated!')

In [ ]:
# Confusion Matrix
print('Generating confusion matrix...')
cm = confusion_matrix(y_test_labels, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True,
            xticklabels=class_folders, yticklabels=class_folders, cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print('Classification Report\n')
print('='*80)
report = classification_report(y_test_labels, y_pred, target_names=class_folders)
print(report)
print('='*80)

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.ravel()

for idx in range(16):
    axes[idx].imshow(X_test[idx])
    true_class = class_folders[y_test_labels[idx]]
    pred_class = class_folders[y_pred[idx]]
    confidence = np.max(y_pred_probs[idx])
    
    color = 'green' if y_pred[idx] == y_test_labels[idx] else 'red'
    axes[idx].set_title(f'True: {true_class.split("___")[-1]}\nPred: {pred_class.split("___")[-1]} ({confidence:.2f})',
                       color=color, fontsize=9, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Save Model

In [ ]:
# Save final model
model_path = os.path.join(MODEL_SAVE_PATH, 'plant_disease_detection_final.h5')
model.save(model_path)
print(f'Model saved to {model_path}')

# Also save best model location
best_model_path = os.path.join(MODEL_SAVE_PATH, 'best_model.h5')
print(f'Best model saved to {best_model_path}')

## 9. Inference Pipeline

In [ ]:
# Load trained model
trained_model = tf.keras.models.load_model(best_model_path)
print('Model loaded successfully!')

In [ ]:
def predict_disease(image_path, model, class_names, top_k=3):
    """
    Predict disease for a single image
    """
    # Load and preprocess image
    image = load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    image_array = img_to_array(image) / 255.0
    image_batch = np.expand_dims(image_array, axis=0)
    
    # Predict
    predictions = model.predict(image_batch, verbose=0)[0]
    
    # Get top-k predictions
    top_k_indices = np.argsort(predictions)[-top_k:][::-1]
    
    # Display results
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    # Show image
    ax[0].imshow(image)
    ax[0].set_title('Input Image')
    ax[0].axis('off')
    
    # Show predictions
    top_classes = [class_names[i].split('___')[-1] for i in top_k_indices]
    top_scores = [predictions[i] for i in top_k_indices]
    
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    ax[1].barh(range(top_k), top_scores, color=colors[:top_k])
    ax[1].set_yticks(range(top_k))
    ax[1].set_yticklabels(top_classes)
    ax[1].set_xlabel('Confidence Score')
    ax[1].set_title(f'Top-{top_k} Predictions')
    ax[1].set_xlim([0, 1])
    
    for i, (class_name, score) in enumerate(zip(top_classes, top_scores)):
        ax[1].text(score + 0.02, i, f'{score:.4f}', va='center')
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nTop {top_k} Predictions:')
    for i, (idx, score) in enumerate(zip(top_k_indices, top_scores)):
        print(f'{i+1}. {class_folders[idx]}: {score:.4f} ({score*100:.2f}%)')
    
    return top_k_indices, top_scores

print('Prediction function defined!')

In [ ]:
# Example: Predict on test image
# Find a test image from dataset
test_class_path = os.path.join(dataset_dir, class_folders[0])
test_images = [f for f in os.listdir(test_class_path) 
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if test_images:
    test_image_path = os.path.join(test_class_path, test_images[0])
    print(f'Predicting on: {class_folders[0]}')
    print(f'Image: {test_images[0]}')
    top_indices, top_scores = predict_disease(test_image_path, trained_model, class_folders)

## 10. Summary and Insights

In [ ]:
print('\n' + '='*80)
print('PROJECT SUMMARY')
print('='*80)

print(f'\n📊 Dataset:')
print(f'  • Total Classes: {len(class_folders)}')
print(f'  • Total Images: {len(images)}')
print(f'  • Image Size: {IMG_HEIGHT}×{IMG_WIDTH}×3')

print(f'\n📈 Model Architecture:')
print(f'  • Type: Convolutional Neural Network')
print(f'  • Total Parameters: {model.count_params():,}')
print(f'  • Conv Blocks: 3')
print(f'  • Dense Layers: 2')

print(f'\n📚 Training Results:')
print(f'  • Training Accuracy: {history.history["accuracy"][-1]:.4f}')
print(f'  • Validation Accuracy: {history.history["val_accuracy"][-1]:.4f}')
print(f'  • Test Accuracy: {test_acc:.4f}')
print(f'  • Test Top-5 Accuracy: {test_top5:.4f}')
print(f'  • Epochs Trained: {len(history.history["loss"])}')

print(f'\n💾 Model Saved:')
print(f'  • Best Model: {best_model_path}')
print(f'  • Final Model: {model_path}')

print(f'\n✨ Next Steps:')
print(f'  • Deploy model using FastAPI/Flask')
print(f'  • Create mobile app wrapper')
print(f'  • Fine-tune on custom dataset')
print(f'  • Implement real-time camera inference')

print('\n' + '='*80)